# Data merge and feature engineering


In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Uw11\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [3]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [6]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
257587,2023-05-17 06:00:00,22,49.4168,26.9743,17.3,8.7,57.6,0.200,100.0,8.33,0.0,0.0,38.9,20.5,152.7,1014.9,92.7,173.9,15.1,6.0,0.92,14.8,14.8,60.37,0.0,0.0,0.0,32.4,16.9,150.8,1014.0,100.0,3.3,0.0,0.0,False,False,True,False,False,True,False,False,2023,5,2,6,Khmelnytskyi,Europe/Kiev,2023-05-17,05:26:22,20:51:34,06:00:00,Хмельницька,0,0.0
124498,2022-09-28 04:00:00,13,49.8444,24.0254,11.6,8.1,80.9,4.200,100.0,8.33,0.0,0.0,37.8,16.2,252.7,1001.4,63.7,113.0,9.7,6.0,0.07,10.4,10.4,92.89,0.0,0.0,0.0,10.1,4.3,168.9,999.0,93.5,0.0,0.1,0.0,False,False,True,False,False,True,False,False,2022,9,2,4,Lviv,Europe/Kiev,2022-09-28,07:18:56,19:09:24,04:00:00,Львівська,0,0.0
34915,2022-04-25 15:00:00,22,49.4168,26.9743,13.7,8.1,70.6,7.000,100.0,8.33,0.0,0.0,29.9,14.4,185.1,1006.9,88.0,181.9,15.9,8.0,0.80,18.6,18.6,43.36,0.0,0.0,0.0,19.8,3.6,130.0,1005.1,90.0,771.0,2.8,8.0,False,False,True,False,False,True,False,False,2022,4,0,15,Khmelnytskyi,Europe/Kiev,2022-04-25,06:01:17,20:19:56,15:00:00,Хмельницька,0,0.0
181585,2023-01-05 07:00:00,3,50.7469,25.3263,6.5,4.0,84.4,7.124,100.0,8.33,0.0,0.0,53.6,35.8,255.4,1006.1,99.5,5.6,0.5,0.0,0.42,7.2,3.1,81.24,0.0,0.0,0.0,51.5,27.0,244.9,1006.0,100.0,0.0,0.1,0.0,False,False,True,False,False,True,False,False,2023,1,3,7,Lutsk,Europe/Kiev,2023-01-05,08:20:08,16:28:05,07:00:00,Волинська,0,0.0
534965,2024-09-09 21:00:00,7,48.6264,22.2851,19.6,8.7,51.7,2.148,100.0,20.83,0.0,0.0,42.1,17.9,88.7,1009.3,68.5,177.5,15.3,6.0,0.21,20.8,20.8,59.44,0.0,0.0,0.0,16.6,0.0,319.0,1005.7,100.0,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2024,9,0,21,Uzhgorod,Europe/Uzhgorod,2024-09-09,06:59:34,19:55:44,21:00:00,Закарпатська,0,0.0
215099,2023-03-04 11:00:00,14,46.9734,31.9852,4.1,-0.3,75.5,0.000,0.0,0.00,0.0,0.0,44.6,23.4,234.9,1011.6,26.6,164.1,14.2,6.0,0.40,6.7,4.5,56.67,0.0,0.0,0.0,35.3,10.8,220.0,1014.0,50.0,530.5,1.9,5.0,False,True,False,False,False,True,False,False,2023,3,5,11,Mykolaiv,Europe/Kiev,2023-03-04,06:27:11,17:41:19,11:00:00,Миколаївська,0,0.0
101090,2022-08-18 13:00:00,4,48.4753,35.0160,24.0,19.0,75.0,0.500,100.0,12.50,0.0,0.0,20.5,14.4,257.8,1008.7,39.2,191.5,16.5,8.0,0.68,28.6,29.1,50.42,0.1,100.0,0.0,20.2,7.2,244.9,1008.0,58.0,770.0,2.8,8.0,False,False,True,False,False,False,True,False,2022,8,3,13,Dnipro,Europe/Kiev,2022-08-18,05:37:22,19:49:23,13:00:00,Дніпропетровська,0,0.0
208808,2023-02-21 13:00:00,10,50.4506,30.5243,3.8,1.1,82.6,9.000,100.0,8.33,0.3,0.9,73.4,33.5,288.2,996.0,78.9,76.2,6.6,3.0,0.04,5.6,0.9,73.84,0.0,0.0,0.0,68.8,29.2,286.9,994.0,99.2,338.8,1.2,3.0,False,False,False,True,False,True,False,False,2023,2,1,13,Kyiv,Europe/Kiev,2023-02-21,06:58:49,17:25:09,13:00:00,Київська,0,0.0
217922,2023-03-09 09:00:00,4,48.4753,35.0160,9.9,4.9,73.9,0.500,100.0,8.33,0.0,0.0,39.6,25.2,197.8,1003.8,61.6,138.1,12.0,6.0,0.57,5.0,1.0,97.93,0.0,0.0,0.0,35.6,19.8,192.7,1005.0,100.0,131.7,0.5,1.0,False,False,True,False,False,True,False,False,2023,3,3,9,Dnipro,Europe/Kiev,2023-03-09,06:06:26,17:35:32,09:00:00,Дніпропетровська,0,0.0
417228,2024-02-18 10:00:00,15,46.4725,30.7371,3.1,1.7,90.7,0.600,100.0,12.50,0.0,0.0,30.6,17.6,305.5,1028.0,96.2,39.3,3.5,2.0,0.29,3.8,0.9,90

In [7]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.4,89.80,0.0,0.0,0.0,16.6,8.3,289.2,1021.0,97.9,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,1,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,01:00:00,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.0,-0.3,90.44,0.0,0.0,0.0,27.7,7.6,287.6,1021.0,98.2,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,2,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,02:00:00,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.9,-0.3,91.75,0.0,0.0,0.0,15.1,7.2,288.9,1021.0,98.8,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,3,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,03:00:00,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.8,-0.2,92.40,0.0,0.0,0.0,13.7,6.5,290.4,1021.0,100.0,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,4,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,04:00:00,Вінницька,0,0.0


In [8]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634680 entries, 0 to 634679
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  634680 non-null  datetime64[us]
 1   region_id                      634680 non-null  int64         
 2   city_latitude                  634680 non-null  float64       
 3   city_longitude                 634680 non-null  float64       
 4   day_temp                       634680 non-null  float64       
 5   day_dew                        634680 non-null  float64       
 6   day_humidity                   634680 non-null  float64       
 7   day_precip                     634680 non-null  float64       
 8   day_precipprob                 634680 non-null  float64       
 9   day_precipcover                634680 non-null  float64       
 10  day_snow                       634680 non-null  float64       
 11  day_snowdepth   

In [9]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)
#df_final['alarm_minutes_lag1'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))
#df_final['alarm_minutes_lag3'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(3).fillna(0))

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
259282,2023-05-20 05:00:00,13,49.8444,24.0254,17.2,11.7,73.2,0.100,100.0,4.17,0.0,0.0,33.1,15.5,68.6,1024.3,47.0,254.6,22.1,7.0,0.02,12.0,12.0,96.12,0.0,0.0,0.0,13.0,5.8,32.4,1025.0,73.2,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2023,5,5,5,Lviv,Europe/Kiev,2023-05-20,05:32:42,21:09:03,05:00:00,Львівська,0,0.000000,0.0,0.0,0.0,0.0,1.0,1,1,3.0
473668,2024-05-26 11:00:00,6,50.2536,28.6654,17.7,4.1,45.7,0.000,0.0,0.00,0.0,0.0,29.5,11.9,75.8,1023.5,29.6,327.0,28.3,9.0,0.60,23.3,23.3,26.70,0.0,0.0,0.0,22.3,9.4,66.3,1024.0,0.0,737.4,2.7,7.0,False,True,False,False,True,False,False,False,2024,5,6,11,Zhytomyr,Europe/Kiev,2024-05-26,05:04:59,21:00:44,11:00:00,Житомирська,0,0.000000,1.0,0.0,1.0,0.0,7.0,1,0,24.0
472583,2024-05-24 13:00:00,26,50.4506,30.5243,20.8,10.6,54.6,0.400,100.0,4.17,0.0,0.0,43.9,18.7,94.2,1023.2,32.9,293.2,25.1,8.0,0.53,26.1,26.1,37.54,0.0,0.0,0.0,43.9,15.8,97.9,1024.0,14.0,837.1,3.0,8.0,False,False,True,False,True,False,False,False,2024,5,4,13,Kyiv,Europe/Kiev,2024-05-24,04:58:43,20:51:45,13:00:00,Київ,1,22.933333,0.0,0.0,0.0,0.0,2.0,0,0,6.0
275329,2023-06-17 02:00:00,3,50.7469,25.3263,18.5,16.7,89.5,9.774,100.0,8.33,0.0,0.0,22.7,13.3,180.8,1010.7,98.4,141.5,12.2,4.0,0.97,16.8,16.8,98.11,0.0,0.0,0.0,15.5,13.3,123.5,1010.0,99.9,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2023,6,5,2,Lutsk,Europe/Kiev,2023-06-17,05:05:06,21:34:16,02:00:00,Волинська,0,0.000000,0.0,0.0,0.0,0.0,2.0,1,1,1.0
540548,2024-09-19 13:00:00,23,49.4407,32.0637,16.7,3.2,44.3,0.000,0.0,0.00,0.0,0.0,38.9,16.9,64.2,1026.0,30.6,198.3,17.1,7.0,0.54,23.3,23.3,21.08,0.0,0.0,0.0,38.2,16.6,76.3,1027.0,37.7,654.7,2.4,7.0,False,True,False,False,False,True,False,False,2024,9,3,13,Cherkasy,Europe/Kiev,2024-09-19,06:34:12,18:55:47,13:00:00,Черкаська,0,0.000000,0.0,0.0,0.0,1.0,5.0,0,0,8.0
109509,2022-09-02 03:00:00,24,48.2924,25.9355,13.9,12.2,89.8,16.042,100.0,41.67,0.0,0.0,21.7,10.9,143.5,1021.0,99.0,27.0,2.3,1.0,0.19,14.8,14.8,86.86,0.0,0.0,0.0,12.7,3.8,336.0,1019.4,100.0,0.0,0.1,0.0,False,False,True,False,False,True,False,False,2022,9,4,3,Chernivtsi,Europe/Kiev,2022-09-02,06:34:51,19:56:14,03:00:00,Чернівецька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,5.0
281806,2023-06-28 07:00:00,25,51.4937,31.2944,16.3,12.3,78.9,11.000,100.0,8.33,0.0,0.0,63.0,28.4,243.5,1005.0,81.3,227.7,19.8,7.0,0.33,14.0,14.0,97.44,0.0,0.0,0.0,14.4,10.1,211.8,1004.0,100.0,20.1,0.1,0.0,False,False,True,False,False,True,False,False,2023,6,2,7,Chernihiv,Europe/Kiev,2023-06-28,04:40:02,21:15:55,07:00:00,Чернігівська,0,0.000000,0.0,0.0,0.0,0.0,2.0,0,0,0.0
322113,2023-09-06 07:00:00,11,48.5085,32.2656,19.5,10.6,60.5,0.000,0.0,0.00,0.0,0.0,31.0,21.6,353.1,1021.7,35.4,221.3,19.2,7.0,0.71,14.2,14.2,78.42,0.0,0.0,0.0,20.9,12.2,353.3,1023.0,12.8,18.2,0.1,0.0,False,True,False,False,True,False,False,False,2023,9,2,7,Kropyvnytskyi,Europe/Kiev,2023-09-06,06:14:27,19:23:33,07:00:00,Кіровоградська,0,0.000000,1.0,1.0,0.0,1.0,5.0,0,0,24.0
356554,2023-11-05 02:00:00,13,49.8444,24.0254,9.1,6.0,81.1,3.000,100.0,4.17,0.0,0.0,57.2,25.2,177.2,99

In [10]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [11]:
def hours_since_last_alarm(series):
    shifted = series.shift(1).fillna(0)
    result = []
    count = 0
    for val in shifted:
        if val == 1:
            count = 0
        else:
            count += 1
        result.append(count)
    return pd.Series(result, index=series.index)

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(hours_since_last_alarm))

In [12]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
281586,2024-02-08 02:00:00,13,49.8444,24.0254,0.8,0.3,96.0,8.000,100.0,8.33,2.3,0.5,35.3,17.6,240.6,1006.1,100.0,19.8,1.7,1.0,0.95,1.4,-3.2,93.73,0.0,0.0,0.0,33.5,17.6,290.3,1005.0,100.0,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2024,2,3,2,Lviv,Europe/Kiev,2024-02-08,07:47:18,17:29:34,02:00:00,Львівська,0,0.0,0.0,0.0,0.0,0.0,4.0,0,1,6.0,0.0,17
213460,2022-05-14 05:00:00,10,50.4506,30.5243,15.5,6.7,57.2,0.100,100.0,4.17,0.0,0.0,45.0,16.6,297.9,1011.5,82.2,197.2,16.7,8.0,0.44,12.5,12.5,77.64,0.0,0.0,0.0,17.3,5.8,233.0,1011.0,76.1,0.0,0.1,0.0,False,False,True,False,False,True,False,False,2022,5,5,5,Kyiv,Europe/Kiev,2022-05-14,05:12:02,20:37:28,05:00:00,Київська,0,0.0,0.0,0.0,0.0,0.0,3.0,1,1,0.0,0.0,6
120096,2023-10-13 14:00:00,6,50.2536,28.6654,14.2,9.9,76.9,4.000,100.0,4.17,0.0,0.0,28.8,13.0,281.8,1017.8,60.5,111.4,9.6,4.0,0.96,18.0,18.0,52.35,0.0,0.0,0.0,25.2,12.6,295.3,1019.0,4.2,438.2,1.6,4.0,False,False,True,False,True,False,False,False,2023,10,4,14,Zhytomyr,Europe/Kiev,2023-10-13,07:23:36,18:18:56,14:00:00,Житомирська,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,3.0,0.0,178
166926,2023-02-03 01:00:00,8,47.8289,35.1626,0.9,-0.7,89.4,0.400,100.0,16.67,0.0,1.3,26.3,16.2,167.6,1014.5,70.7,82.1,7.0,4.0,0.42,0.4,-3.9,92.32,0.0,0.0,0.0,22.3,14.8,183.5,1015.0,100.0,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2023,2,4,1,Zaporozhye,Europe/Zaporozhye,2023-02-03,07:04:34,16:42:22,01:00:00,Запорізька,0,0.0,0.0,0.0,0.0,1.0,3.0,0,1,0.0,0.0,11
388238,2024-03-15 10:00:00,17,50.6193,26.2513,3.3,0.6,83.8,0.000,0.0,0.00,0.0,0.0,25.2,10.8,154.0,1020.0,52.2,144.7,12.4,5.0,0.17,5.5,4.0,79.29,0.0,0.0,0.0,17.3,6.8,120.6,1021.0,45.2,331.7,1.2,3.0,False,True,False,False,False,True,False,False,2024,3,4,10,Rivne,Europe/Kiev,2024-03-15,06:28:05,18:20:30,10:00:00,Рівненська,0,0.0,0.0,0.0,0.0,0.0,2.0,0,0,3.0,0.0,20
225224,2023-09-16 10:00:00,10,50.4506,30.5243,16.8,13.2,80.8,0.800,100.0,4.17,0.0,0.0,23.0,11.9,48.0,1025.8,88.3,95.3,8.1,3.0,0.05,15.0,15.0,84.45,0.0,0.0,0.0,18.7,8.3,61.6,1026.0,90.6,132.9,0.5,1.0,False,False,True,False,False,True,False,False,2023,9,5,10,Kyiv,Europe/Kiev,2023-09-16,06:34:17,19:10:38,10:00:00,Київська,0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,3.0,0.0,150
495719,2024-05-25 08:00:00,21,46.6401,32.6142,19.6,3.5,36.6,0.000,0.0,0.00,0.0,0.0,52.2,25.9,62.6,1021.0,2.1,332.3,28.6,9.0,0.56,18.7,18.7,34.93,0.0,0.0,0.0,40.3,22.3,59.2,1022.0,0.0,313.2,1.1,3.0,True,False,False,False,True,False,False,False,2024,5,5,8,Kherson,Europe/Kiev,2024-05-25,05:04:50,20:28:57,08:00:00,Херсонська,0,0.0,0.0,0.0,0.0,1.0,8.0,1,0,0.0,0.0,6
422316,2025-01-27 12:00:00,18,50.9080,34.7972,4.4,3.6,94.6,0.115,100.0,8.33,0.0,0.0,39.6,23.4,193.8,1020.6,100.0,14.4,1.4,1.0,0.93,5.1,1.5,93.25,0.0,0.0,0.0,34.2,16.9,206.2,1021.0,100.0,69.8,0.3,1.0,False,False,True,False,False,True,False,False,2025,1,0,12,Sumy,Europe/Kiev,2025-01-27,07:24:12,16:23:41,12:00:00,Сумська,1,60.0,1.0,1.0,0.0,1.0,20.0,0,0,2.0,0.0,0
465793,2024-01-01 06:00:00,20,50.0042,36.2358,1.0,-1.2,85.4,0.300,100.0,12.5

In [13]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
165,2022-08-07,0.704463,-0.223520,0.217479,-0.064640,-0.076640,0.113478,-0.001755,0.068555,0.155479,-0.129257,0.047586,-0.005814,-0.029551,-0.163066,-0.092632,0.078703,-0.084993,0.042595,-0.109730,-0.040833,-0.022201,0.070454,0.002449,-0.139510,0.077842,-0.050667,-0.071332,-0.033498,-0.007079,-0.063468,0.018467,0.032779,-0.021615,-0.022973,0.000698,0.010115,0.004603,0.014994,-0.032008,-0.020954,0.007831,0.000542,0.033269,0.000414,0.047826,-0.028002,-0.027318,0.030428,0.012277,-0.054936,0.014533,0.047379,0.003866,0.000253,-0.004247,-0.034601,0.004392,-0.021009,0.002891,0.015253,-0.032777,0.003372,-0.019544,0.001574,0.045949,-0.011345,-0.008872,-0.016711,-0.043395,0.008829,0.007191,-0.041674,-0.009121,-0.003429,-0.030518,0.024193,-0.042378,0.039976,-0.018245,-0.003833,-0.017614,-0.008889,-0.054812,0.018619,0.000525,0.027568,-0.001807,-0.011538,-0.027853,0.036587,0.020848,-0.034160,0.008828,-0.022065,-0.018241,0.003424,-0.006993,-0.007278,0.041904,0.004164,0.026395,-0.001777,0.024665,0.010205,0.021680,-0.037459,-0.018406,-0.033702,0.010960,0.031496,0.018253,0.030395,0.011274,-0.014571,-0.018441,0.023158,-0.011148,-0.017726,0.021447,-0.010866,-0.007156,-0.005360,0.020055,-0.010297,-0.004697,0.028017,0.008439,-0.001740,0.016943,0.027491,-0.010272,0.012294,-0.054201,-0.025263,-0.019919,0.021211,-0.021132,-0.034642,-0.020175,0.013086,0.019968,-0.027974,-0.003742,-0.035248,0.012605,-0.006451,0.014536,-0.035769,0.005223,-0.040830
474,2023-06-15,0.810533,-0.199448,0.022822,-0.067516,-0.089837,0.080004,0.032868,-0.023151,-0.074613,0.078943,-0.017435,0.229366,-0.060342,0.068703,0.001954,-0.126380,-0.090490,0.059556,0.031597,0.058231,0.019373,0.020109,0.000108,-0.026535,-0.041180,-0.076606,-0.039198,0.012435,0.030682,0.024514,-0.003271,-0.003149,0.016733,0.027025,0.028908,0.005084,-0.031863,-0.012771,-0.019409,-0.006814,-0.017268,-0.050022,-0.008876,-0.024673,0.062032,-0.013703,0.038274,0.004608,0.001375,0.044383,0.002159,0.017072,-0.026321,-0.020086,0.003281,0.027738,0.029937,0.028

In [14]:
#df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)    ТУТ ЗСУВ ІСВ НА + 1 ДЕНЬ
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [15]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [16]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [17]:
df_final.head(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0,0.0,0.0,0.0,0.0,NaN,0,1,NaN,NaN,1,0.648336,-0.079572,0.178041,0.061854,0.073453,-0.055809,-0.011311,0.047310,0.008354,0.006474,-0.047099,0.020496,0.050190,-0.038573,0.057228,-0.072471,0.045393,-0.116783,0.168249,0.167355,-0.025072,-0.073858,-0.097024,-0.146191,0.027983,0.082989,0.014423,0.058370,-0.033928,0.046265,0.006901,-0.049498,0.126117,-0.089039,-0.006767,-0.031686,-0.010346,0.025637,0.061278,0.162580,-0.028081,0.161291,-0.018217,-0.002657,0.038485,-0.000530,0.044413,0.107176,-0.050721,0.064279,-0.066677,0.016742,-0.050257,0.072549,-0.035201,-0.027279,0.017115,0.043631,0.024660,-0.025841,0.031320,-0.076959,-0.072986,-0.040465,0.003958,-0.022475,0.033208,-0.028056,-0.0054

In [18]:
df_final.isna().sum()

datetime_hour        0
region_id            0
city_latitude        0
city_longitude       0
day_temp             0
                  ... 
isw_topic_145     5760
isw_topic_146     5760
isw_topic_147     5760
isw_topic_148     5760
isw_topic_149     5760
Length: 216, dtype: int64

In [19]:
df_final.fillna(0, inplace=True)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,1,0.648336,-0.079572,0.178041,0.061854,0.073453,-0.055809,-0.011311,0.047310,0.008354,0.006474,-0.047099,0.020496,0.050190,-0.038573,0.057228,-0.072471,0.045393,-0.116783,0.168249,0.167355,-0.025072,-0.073858,-0.097024,-0.146191,0.027983,0.082989,0.014423,0.058370,-0.033928,0.046265,0.006901,-0.049498,0.126117,-0.089039,-0.006767,-0.031686,-0.010346,0.025637,0.061278,0.162580,-0.028081,0.161291,-0.018217,-0.002657,0.038485,-0.000530,0.044413,0.107176,-0.050721,0.064279,-0.066677,0.016742,-0.050257,0.072549,-0.035201,-0.027279,0.017115,0.043631,0.024660,-0.025841,0.031320,-0.076959,-0.072986,-0.040465,0.003958,-0.022475,0.033208,-0.028056,-0

In [20]:
df_final.isna().sum()

datetime_hour     0
region_id         0
city_latitude     0
city_longitude    0
day_temp          0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [21]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = (
    df_final.groupby('region_id')['isw_total_intensity']
    .diff(24)
    .fillna(0)
)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\1269306822.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\1269306822.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\1269306822.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [22]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
137083,2022-09-08 11:00:00,7,48.62636,22.28514,20.6,14.2,68.9,0.008,100.0,4.17,0.0,0.0,46.8,13.7,82.9,1014.9,61.9,189.5,16.3,7.0,0.39,21.2,21.2,65.53,0.0,0.0,0.0,33.8,7.9,99.0,1015.8,34.8,479.0,1.7,5.0,False,False,True,False,False,True,False,False,2022,9,3,11,Uzhgorod,Europe/Uzhgorod,2022-09-08,06:57:24,19:58:53,11:00:00,Закарпатська,0,0.000000,0.0,0.0,0.0,0.0,1.0,0,0,1.0,0.0,15,0.809727,-0.221686,0.154321,-0.152486,-0.026332,0.033961,-0.024565,0.017441,0.050666,-0.078529,0.006984,0.021928,0.136802,0.103640,-0.019146,0.023123,-0.107389,0.093725,0.009609,0.081766,0.043083,0.031293,0.021245,0.035246,0.074513,0.036859,-0.032635,-0.058497,-0.031134,0.022858,0.021038,-0.008669,0.061904,0.029781,0.008352,0.020688,0.033577,-0.001511,-0.064946,-0.022478,-0.019356,0.050694,0.044830,-0.009204,0.044311,0.011212,0.028720,-0.065306,-0.018759,-0.006198,0.033734,-0.000747,-0.013840,-0.018094,-

### TELEGRAM MERGE

In [23]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [24]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,0.042624,-0.024341,-0.018764,0.023404,-0.017137,-0.358836,0.425288,0.017966,-0.107660,-0.032066,-0.045986,0.118212,0.091172,-0.248637,-0.001069,0.150763,0.030603,-0.074030,0.005803,0.005871,0.034871,0.017239,0.058098,0.029814,-0.010845,0.031486,-0.022705,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005139,0.007857,0.016734,-0.040289,-0.019688,-0.026763,0.009381,-0.043500,-0.009525,-0.013075,-0.010473,-0.018150,0.005988,0.032611,0.000850,0.017872,0.020885,0.023899,-0.019290,-0.006676,-0.018478,-0.023263,-0.005497,-0.006460,0.026051,0.004991,0.019687,-0.022244,0.010929,0.007016,-0.067349,-0.035249,-0.005686,0.018747,0.039159,-0.053868,-0.078189,-0.074737,-0.082163,0.020498,0.084336,0.113470,-0.025871,-0.048314,0.096

In [25]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [26]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]

tg_hourly = (
    df_tg_matrix.groupby("datetime_hour")[topic_cols]
    .mean()
    .sort_index()
    .reset_index()
)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\2188712141.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


In [27]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,0.042624,-0.024341,-0.018764,0.023404,-0.017137,-0.358836,0.425288,0.017966,-0.107660,-0.032066,-0.045986,0.118212,0.091172,-0.248637,-0.001069,0.150763,0.030603,-0.074030,0.005803,0.005871,0.034871,0.017239,0.058098,0.029814,-0.010845,0.031486,-0.022705,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005139,0.007857,0.016734,-0.040289,-0.019688,-0.026763,0.009381,-0.043500,-0.009525,-0.013075,-0.010473,-0.018150,0.005988,0.032611,0.000850,0.017872,0.020885,0.023899,-0.019290,-0.006676,-0.018478,-0.023263,-0.005497,-0.006460,0.026051,0.004991,0.019687,-0.022244,0.010929,0.007016,-0.067349,-0.035249,-0.005686,0.018747,0.039159,-0.053868,-0.078189,-0.074737,-0.082163,0.020498,0.084336,0.113470,-0.025871,-

In [28]:
#tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)        ТУТ ЗСУВ ТГ НА 1 ГОДИНУ 

In [29]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [30]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [31]:
df_final.shape

(635256, 473)

In [32]:
df_final = df_final.sort_values(["datetime_hour"])

In [33]:
with pd.option_context('display.max_rows', None):
    print(df_final.isnull().sum())

datetime_hour                         0
region_id                             0
city_latitude                         0
city_longitude                        0
day_temp                              0
day_dew                               0
day_humidity                          0
day_precip                            0
day_precipprob                        0
day_precipcover                       0
day_snow                              0
day_snowdepth                         0
day_windgust                          0
day_windspeed                         0
day_winddir                           0
day_pressure                          0
day_cloudcover                        0
day_solarradiation                    0
day_solarenergy                       0
day_uvindex                           0
day_moonphase                         0
hour_temp                             0
hour_feelslike                        0
hour_humidity                         0
hour_precip                           0


In [34]:
df_final = df_final.fillna(0)

In [35]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\409095542.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\409095542.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\409095542.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [36]:
df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
df_final['tg_intensity_zscore'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: (x - x.rolling(24, min_periods=1).mean()) / (x.rolling(24, min_periods=1).std() + 1e-9)))

C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\3857173777.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\3857173777.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_24900\3857173777.py:3: PerformanceWa

In [37]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [38]:
df_final.to_parquet("merged_dataset.parquet", index=False, engine="pyarrow")

# ТУТ БУДЕ EDA

## Weather, ISW, Telegram shift for further models trainings

In [39]:
df_to_train = df_final.copy()
df_to_train = df_to_train.sort_values(['region_id', 'datetime_hour'])

In [40]:
# delete unnessecary columns
columns_to_delete = ['city_latitude', 'city_longitude', 'day_temp', 'day_dew', 'day_humidity', 'day_precip', 'day_precipprob', 'day_precipcover', 
                     'day_snow', 'day_snowdepth', 'day_windgust', 'day_windspeed', 'day_winddir', 'day_pressure', 'day_cloudcover', 'day_solarradiation',
                     'day_solarenergy', 'day_uvindex', 'day_moonphase', 'day_conditions_simple_Clear', 'day_conditions_simple_Cloudy', 'day_conditions_simple_Rain',
                     'day_conditions_simple_Snow', 'year', 'month', 'hour', 'day_of_week', 'city_timezone', 'day_datetime', 'hour_datetime', 'region_key']
df_to_train = df_to_train.drop(columns=columns_to_delete)

In [41]:
isw_cols = [c for c in df_to_train.columns if 'isw_' in c]
df_to_train[isw_cols] = df_to_train.groupby('region_id')[isw_cols].shift(24).fillna(0)

In [42]:
df_to_train.head()

,datetime_hour,region_id,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,city_name,day_sunrise,day_sunset,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,t

In [43]:
tg_cols = [c for c in df_to_train.columns if 'tg_' in c]
df_to_train[tg_cols] = df_to_train.groupby('region_id')[tg_cols].shift(1).fillna(0)

In [44]:
hour_weather_cols = [c for c in df_to_train.columns if c.startswith('hour_')]
df_to_train[hour_weather_cols] = (df_to_train.groupby('region_id')[hour_weather_cols].shift(1))
df_to_train[hour_weather_cols] = (df_to_train.groupby('region_id')[hour_weather_cols].transform(lambda x: x.bfill()))

In [45]:
df_to_train['alarm_minutes_in_hour'] = (df_to_train.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))

In [46]:
df_to_train.head()

,datetime_hour,region_id,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,city_name,day_sunrise,day_sunset,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,t